In [1]:
import sys
import numpy as np
import pandas as pd

from one.api import ONE
from brainbox.io.one import SessionLoader

from brainwidemap.bwm_loading import load_good_units, load_trials_and_mask, merge_probes

from brainwidemap.decoding.functions.decoding import fit_eid
from brainwidemap.decoding.functions.process_targets import load_behavior
from brainwidemap.decoding.settings_template import params
from brainwidemap.bwm_loading import bwm_query, load_good_units, load_trials_and_mask
from brainbox.io.one import SessionLoader
from brainwidemap.bwm_loading import load_trials_and_mask
from prior_localization.functions.utils import compute_mask
import tempfile
import pickle
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from one.api import ONE
from prior_localization.fit_data import fit_session_ephys, fit_session_widefield
from brainwidemap.decoding.functions.process_inputs import select_ephys_regions
from brainwidemap.decoding.functions.process_inputs import preprocess_ephys
from brainwidemap.decoding.functions.process_targets import compute_beh_target
from brainwidemap.decoding.functions.process_targets import compute_target_mask
from brainwidemap.decoding.functions.process_targets import transform_data_for_decoding
from brainwidemap.decoding.functions.process_targets import logisticreg_criteria
from brainwidemap.decoding.functions.process_targets import get_target_data_per_trial_wrapper

from prior_localization.prepare_data import (
    prepare_ephys,
    prepare_behavior,
    prepare_motor,
    prepare_pupil,
    prepare_widefield,
    prepare_widefield_old,
)
from prior_localization.fit_data import fit_target

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org", password="international", silent=True
)

In [4]:
sessions = one.search(datasets="widefieldU.images.npy")
print(f"{len(sessions)} sessions with widefield data found")

50 sessions with widefield data found


In [5]:
session_id = sessions[0]
sl = SessionLoader(one, eid=session_id)
sl.load_trials()
trials_mask = compute_mask(
    sl.trials, align_event="stimOn_times", min_rt=0.08, max_rt=None, n_trials_crop_end=1
)
pseudo_ids = np.array([-1])
params["target"] = "pLeft"
subject = one.eid2ref(session_id)["subject"]
output_dir = Path(tempfile.TemporaryDirectory().name)

In [6]:
choice_target = sl.trials["choice"].values

# Create a mask for valid trials (non-NaN choice)
# You can add valid reaction times or other criteria here if needed
trials_mask = ~np.isnan(choice_target) & (choice_target != 0)

# Apply mask immediately so our target variable is clean
clean_target = choice_target[trials_mask]
clean_trials_df = sl.trials[trials_mask].reset_index(drop=True)

print(f"Valid trials found: {len(clean_target)}")

if len(clean_target) < 50:
    raise ValueError("Not enough valid trials for decoding.")

# 3. Load Widefield Data
print("Loading widefield data...")
# Adjust frame_window as needed (default is -2 to -2 relative to event)
align_event = "firstMovement_times"  # 'stimOn_times' or 'firstMovement_times'

data_epoch, actual_regions = prepare_widefield(
    one,
    session_id,
    regions="all_regions",  # or specify list of regions
    align_times=sl.trials[align_event].values,
    frame_window=(-2, 0),
    hemisphere=("left", "right"),
)

Valid trials found: 929
Loading widefield data...


/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/wfield/utils.py:117: RuntimeWarning: divide by zero encountered in matmul
  res = (nM @ xy).T
/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/wfield/utils.py:117: RuntimeWarning: overflow encountered in matmul
  res = (nM @ xy).T
/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/wfield/utils.py:117: RuntimeWarning: invalid value encountered in matmul
  res = (nM @ xy).T
/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/wfield/utils.py:404: RuntimeWarning: divide by zero encountered in matmul
  t = self.Uflat[idx,:]@self.SVT
/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/wfield/utils.py:404: RuntimeWarning: overflow encountered in matmul
  t = self.Uflat[idx,:]@self.SVT
/Users/dkundu/mamba/envs/info-decom/lib/python3.10/site-packages/wfield/utils.py:404: RuntimeWarning: invalid value encountered in matmul
  t = self.Uflat[idx,:]@self.SVT
/Users/dkundu/Documen

In [7]:
from prior_localization import fit_data
from sklearn.linear_model import LogisticRegression

In [30]:
region_data = data_epoch[0]

# Apply the same mask to the neural data
# region_data is shape (n_trials, n_features)
clean_data = region_data[trials_mask, 0, :]

# Prepare inputs for fit_target
# Argument `all_data` expects a list of arrays (one per pseudo-session).
# We only have one (the actual session), so we wrap in a list [].
input_data = [clean_data[:, 0:1033]]
input_target = [clean_target]
input_trials = [clean_trials_df]

# We pass None for neurometrics to skip that calculation
input_neurometrics = [None]
fit_data.config["estimator"] = LogisticRegression

# Optional: Adjust hyperparameters for Logistic Regression if needed
# (e.g., C values for regularization)
fit_data.config["hparam_grid"] = {"C": [0.001, 0.01, 0.1, 1, 10, 100]}
fit_data.config["estimator_kwargs"] = {"solver": "liblinear", "max_iter": 1000}

In [64]:
clean_data.shape

(929, 2066)

In [ ]:
# Call the decoding function directly
fit_results = fit_target(
    all_data=input_data,
    all_targets=input_target,
    all_trials=input_trials,
    n_runs=10,  # Number of cross-validation repeats
    all_neurometrics=input_neurometrics,
    pseudo_ids=[-1],  # Explicitly state this is the real session
    base_rng_seed=12345,
)

In [54]:
fit_results[3]["acc_test_full"]

0.5468245425188375

In [86]:
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.utils import compute_sample_weight


def fit_new(neural_data, trial_labels, n_bootstraps=5, n_splits=5):
    X = neural_data  # (n_trials, n_neurons)
    y = trial_labels.flatten()

    n_trials, n_neurons = X.shape

    results = []

    print(f"Starting bootstrapping: {n_bootstraps} iterations with {n_splits}-Fold CV...")
    param_grid = {"clf__C": [0.01, 0.1, 1]}
    probs_A_all = np.zeros((n_trials))

    for i in range(n_bootstraps):
        # 2. Sub-sample neurons

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=None)
        best_params = []
        for train_idx, test_idx in skf.split(X, y):
            # Split Data
            X_train_A, X_test_A = X[train_idx], X[test_idx]

            y_train = y[train_idx]
            train_weights = compute_sample_weight(class_weight="balanced", y=y_train)

            # Train Decoders
            steps_A = [("clf", LogisticRegression(solver="liblinear", max_iter=1000))]
            pipeline_A = Pipeline(steps_A)

            # Predict and Fill
            # test_idx here corresponds to the indices in the ORIGINAL X array.
            # So probs_A_all will remain aligned with the original trial order.
            grid_A = GridSearchCV(
                pipeline_A, param_grid, cv=5, scoring="balanced_accuracy", n_jobs=-1
            )

            # Fit the Grid Search (This runs the Inner Loop)
            # Note: We pass sample_weight via the **fit_params specific to the 'clf' step
            grid_A.fit(X_train_A, y_train, clf__sample_weight=train_weights)
            # E. Store Best Params
            best_params.append(grid_A.best_params_["clf__C"])

            # F. Predict on Test Data (Outer Loop)
            # grid.predict_proba automatically uses the best refitted model
            probs_A_all[test_idx] = grid_A.predict(X_test_A)

        # 4. Calculate Final Metrics (Same as before)
        preds_A = probs_A_all

        metrics = {
            "accuracy_A": accuracy_score(y, preds_A),
            "balanced_acc_A": balanced_accuracy_score(y, preds_A),
        }

        run_data = {
            "iteration": i,
            "probs_A": probs_A_all,
        }
        run_data.update(metrics)
        results.append(run_data)

    return results

In [87]:
from ibl_info.decoder_pid import run_decoder_bootstrapping_CV


A = fit_new(
    clean_data,
    clean_target,
    n_bootstraps=1,
    n_splits=5,
)

Starting bootstrapping: 1 iterations with 5-Fold CV...


In [34]:
# now we use mine

In [57]:
import warnings

warnings.filterwarnings("ignore")

Starting bootstrapping: 1 iterations with 5-Fold CV...
Bootstrapping complete.


IndexError: list index out of range

### This is ephys

In [ ]:
bwm_df = bwm_query(one, freeze="2022_10_bwm_release")
idx = bwm_df[bwm_df.eid == eid].index[1]  # take single probe
subject = bwm_df.iloc[idx]["subject"]
pid = bwm_df.iloc[idx]["pid"]
probe_name = bwm_df.iloc[idx]["probe_name"]

"""
--------------------------------
Load data
--------------------------------
"""

# load trials df
sess_loader = SessionLoader(one=one, eid=eid)
sess_loader.load_trials()

# create mask
trials_df, trials_mask = load_trials_and_mask(
    one=one,
    eid=eid,
    sess_loader=sess_loader,
    min_rt=params["min_rt"],
    max_rt=params["max_rt"],
    min_trial_len=params["min_len"],
    max_trial_len=params["max_len"],
    exclude_nochoice=True,
    exclude_unbiased=params["exclude_unbiased_trials"],
)
params["trials_mask_diagnostics"] = [trials_mask]

# load target data if necessary
if params["target"] in ["wheel-vel", "wheel-speed", "l-whisker-me", "r-whisker-me"]:
    raise NotImplementedError(
        "see 04_decode_single_session.py for proper handling of wheel and dlc targets"
    )
else:
    dlc_dict = None
    params["imposter_df"] = None

# Load spike sorting data
spikes, clusters = load_good_units(one, pid, eid=eid, pname=probe_name)

# Put everything into the input format fit_eid still expects at this point
neural_dict = {
    "spk_times": spikes["times"],
    "spk_clu": spikes["clusters"],
    "clu_regions": clusters["acronym"],
    "clu_qc": {k: np.asarray(v) for k, v in clusters.to_dict("list").items()},
    "clu_df": clusters,
}
metadata = {"subject": subject, "eid": eid, "probe_name": probe_name}

In [ ]:
"""Example script that fits decoders for a single eid.

These are snippets of code taken from 04_decode_single_session.py to illustrate a simplified
pipeline. To run from the command line:

```
(iblenv) $ python decoding_example_script.py
```

"""

from brainbox.io.one import SessionLoader
import matplotlib.pyplot as plt
import numpy as np
from one.api import ONE
from pathlib import Path
import pickle

from brainwidemap.bwm_loading import bwm_query, load_good_units, load_trials_and_mask
from brainwidemap.decoding.functions.decoding import fit_eid
from brainwidemap.decoding.functions.utils import get_save_path
from brainwidemap.decoding.settings_template import params


# connect to server
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org", password="international", silent=True
)

"""
--------------------------------
User input
--------------------------------
"""
# where results are saved
# results_dir = Path('/media/mattw/ibl/tmp')
results_dir = Path().home().joinpath("bwm_decoding_example")

# select example eid for decoding analysis
eid = "b658bc7d-07cd-4203-8a25-7b16b549851b"

# perform decoding on original eid (-1 entry) and 5 pseudo-sessions
pseudo_ids = np.array([-1])

# select variable to decode
# targets other than 'pLeft' and 'signcont' (stimulus) require more processing to obtain
# the relevant null distributions that are not supported in this example; see
# 03_decode_single_session.py for more detailed information
params["target"] = "signcont"
params["tanh_transform"] = True  # only True for target=='signcont'

"""
--------------------------------
Update info from user selections
--------------------------------
"""
# update paths
params["behfit_path"] = results_dir.joinpath("decoding", "results", "behavioral")
params["behfit_path"].mkdir(parents=True, exist_ok=True)
params["neuralfit_path"] = results_dir.joinpath("decoding", "results", "neural")
params["neuralfit_path"].mkdir(parents=True, exist_ok=True)
params["add_to_saving_path"] = (
    f"_binsize={1000 * params['binsize']}_lags={params['n_bins_lag']}_mergedProbes_{False}"
)

# get other info from this eid
bwm_df = bwm_query(one, freeze="2022_10_bwm_release")
idx = bwm_df[bwm_df.eid == eid].index[1]  # take single probe
subject = bwm_df.iloc[idx]["subject"]
pid = bwm_df.iloc[idx]["pid"]
probe_name = bwm_df.iloc[idx]["probe_name"]

"""
--------------------------------
Load data
--------------------------------
"""

# load trials df
sess_loader = SessionLoader(one=one, eid=eid)
sess_loader.load_trials()

# create mask
trials_df, trials_mask = load_trials_and_mask(
    one=one,
    eid=eid,
    sess_loader=sess_loader,
    min_rt=params["min_rt"],
    max_rt=params["max_rt"],
    min_trial_len=params["min_len"],
    max_trial_len=params["max_len"],
    exclude_nochoice=True,
    exclude_unbiased=params["exclude_unbiased_trials"],
)
params["trials_mask_diagnostics"] = [trials_mask]

# load target data if necessary
if params["target"] in ["wheel-vel", "wheel-speed", "l-whisker-me", "r-whisker-me"]:
    raise NotImplementedError(
        "see 04_decode_single_session.py for proper handling of wheel and dlc targets"
    )
else:
    dlc_dict = None
    params["imposter_df"] = None

# Load spike sorting data
spikes, clusters = load_good_units(one, pid, eid=eid, pname=probe_name)

# Put everything into the input format fit_eid still expects at this point
neural_dict = {
    "spk_times": spikes["times"],
    "spk_clu": spikes["clusters"],
    "clu_regions": clusters["acronym"],
    "clu_qc": {k: np.asarray(v) for k, v in clusters.to_dict("list").items()},
    "clu_df": clusters,
}
metadata = {"subject": subject, "eid": eid, "probe_name": probe_name}

"""
--------------------------------
Run decoding
--------------------------------
"""
# perform full nested xv decoding
# for pLeft, 5 pseudo-sessions, should take ~1 minute on a cpu
print(f"saving results to {results_dir}")
results_fit_eid = fit_eid(
    neural_dict=neural_dict,
    trials_df=trials_df,
    trials_mask=trials_mask,
    metadata=metadata,
    pseudo_ids=pseudo_ids,
    dlc_dict=dlc_dict,
    **params,
)

In [ ]:
results_fit_eid

In [ ]:
region = "CP"
save_path = get_save_path(
    -1,
    metadata["subject"],
    metadata["eid"],
    "ephys",
    probe=metadata["probe_name"],
    region=region,
    output_path=params["neuralfit_path"],
    time_window=params["time_window"],
    date=params["date"],
    target=params["target"],
    add_to_saving_path=params["add_to_saving_path"],
)
# save_path = results_fit_eid[0]  # can also access results from list returned by fit function
results = pickle.load(open(save_path, "rb"))

In [ ]:
curr_fit = results["fit"][0]

In [ ]:
t = np.where(curr_fit["mask"])[0]
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(t, curr_fit["target"], label="True")
ax.plot(t, curr_fit["predictions_test"], label="Predicted")
ax.set_xlabel("Trial number")
ax.set_ylabel(params["target"])
ax.set_title(
    "eid=%s\nsubject=%s\nregion=%s, probe=%s"
    % (results["eid"], results["subject"], results["region"][0], results["probe"])
)
ax.text(
    0.05, 0.9, "$R^2$=%1.2f" % curr_fit["scores_test_full"], transform=ax.transAxes, fontsize=12
)
ax.legend(loc="upper right")

## New stuff